Step 1: Load the data

In [3]:
import pandas as pd

movies = pd.read_csv("CSV_files/movies_metadata.csv", low_memory=False) # low_memory=False prevents Pandas from incorrectly guessing data types when reading a large file.

movies = movies[['title', 'overview']]

movies.head()

,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...


Step 2: Handle missing values

In [4]:
movies['overview'] = movies['overview'].fillna('')

#The fillna('') method replaces every missing overview with an empty string ('').

#This is necessary because the TF-IDF vectorizer cannot process missing (NaN) values.

Step 3: Convert overviews into TF-IDF vectors

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['overview'])

#TfidfVectorizer converts text into numerical features.
#stop_words='english' removes common words such as "the", "is", and "and" because they provide little useful information.
#fit_transform() : Converts each overview into a numerical vector based on TF-IDF scores.
#tfidf_matrix: each row represents a movie,each column represents a unique word,each value represents the importance of that word in the movie's overview
#Transform textual movie descriptions into numerical representations that can be compared mathematically.

Step 4: Compute cosine similarity

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

#Cosine similarity measures how similar two movies are based on their TF-IDF vectors.
#Calculate the similarity between every pair of movies based on their content

Step 5: Create an index of movie titles

In [7]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

#This creates a mapping between movie titles and their row indices in the DataFrame.
#Allow the recommender to quickly locate a movie using its title.

Step 6: Recommendation function

In [8]:
def content_recommender(title, cosine_sim=cosine_sim):

    idx = indices[title] #Find the movie index

    sim_scores = list(enumerate(cosine_sim[idx])) #Retrieve similarity scores

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True) #Sort by similarity

    sim_scores = sim_scores[1:11] #Remove the selected movie and keep the top 10

    movie_indices = [i[0] for i in sim_scores] #Retrieve movie indices

    return movies.iloc[movie_indices][['title']] # Return the recommendations

#Generate a list of movies with descriptions most similar to the movie selected by the user.

Step 7: Test it

In [9]:
content_recommender("Toy Story")

,title
15348,Toy Story 3
2997,Toy Story 2
10301,The 40 Year Old Virgin
24523,Small Fry
23843,Andy Hardy's Blonde Trouble
29202,Hot Splash
43427,Andy Kaufman Plays Carnegie Hall
38476,Superstar: The Life and Times of Andy Warhol
42721,Andy Peters: Exclamation Mark Question Point
8327,The Champ


In [10]:
content_recommender("Jumanji")

,title
21633,Table No. 21
45253,Quiz
41573,Snowed Under
35509,The Mend
44376,Liar Game: Reborn
17223,The Dark Angel
8801,Quintet
6166,Brainscan
30981,Turkey Shoot
9503,Word Wars
